In [1]:
import nltk
import random
import pandas as pd
from nltk.corpus import movie_reviews
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

# Download required NLTK data
nltk.download('movie_reviews')


[nltk_data] Downloading package movie_reviews to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Unzipping corpora/movie_reviews.zip.


True

In [12]:
# Prepare movie reviews dataset
documents = [(list(movie_reviews.words(fileid)), category)
            for category in movie_reviews.categories()
            for fileid in movie_reviews.fileids(category)]
print(f"Total documents: {len(documents)}")
# Shuffle and prepare data
random.shuffle(documents)
reviews = [" ".join(words) for words, _ in documents]
sentiments = [sentiment for _, sentiment in documents]

# Create a DataFrame for better visualization
df = pd.DataFrame({
    'text': reviews,
    'sentiment': sentiments
})

print("Dataset Overview:")
print(df['sentiment'].value_counts())
print("\nSample Reviews:")
print(df.head())



Total documents: 2000
Dataset Overview:
sentiment
pos    1000
neg    1000
Name: count, dtype: int64

Sample Reviews:
                                                text sentiment
0  it was a rainy friday afternoon in columbus wh...       pos
1  * * * the following review contains spoilers *...       neg
2  from a major league baseball radio broadcast ,...       neg
3  the happy bastard ' s quick movie review me ta...       pos
4  usually a movie is about something more than a...       pos


In [6]:
# Create and configure TF-IDF vectorizer
vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=5,
    max_df=0.7,
    stop_words='english'
)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['sentiment'], 
    test_size=0.2, 
    random_state=42
)

# Transform text data
X_train_tfidf = vectorizer.fit_transform(X_train)



In [7]:
X_test_tfidf = vectorizer.transform(X_test)

# Display feature information
print("\nNumber of features:", len(vectorizer.get_feature_names_out()))
print("Top features:", vectorizer.get_feature_names_out()[:10])



Number of features: 5000
Top features: ['000' '10' '100' '11' '12' '13' '13th' '14' '15' '16']


In [8]:
# Initialize and train the classifier
classifier = LogisticRegression(random_state=42, max_iter=1000)
classifier.fit(X_train_tfidf, y_train)

# Make predictions
y_pred = classifier.predict(X_test_tfidf)

# Print detailed performance metrics
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Print overall accuracy
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")



Classification Report:
              precision    recall  f1-score   support

         neg       0.84      0.85      0.85       200
         pos       0.85      0.83      0.84       200

    accuracy                           0.84       400
   macro avg       0.85      0.84      0.84       400
weighted avg       0.85      0.84      0.84       400

Accuracy: 0.8450


In [9]:
feature_names = vectorizer.get_feature_names_out()
feature_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': abs(classifier.coef_[0])
})


In [10]:
# Sort and display top important features
print("\nTop 10 Most Important Words for Classification:")
print(feature_importance.sort_values('importance', ascending=False).head(10))



Top 10 Most Important Words for Classification:
       feature  importance
372        bad    3.343384
4965     worst    1.989563
3347      plot    1.956390
1970     great    1.647668
2628      life    1.624716
4370  supposed    1.478052
523     boring    1.425136
4850     waste    1.347549
3602    reason    1.316022
3897    script    1.309443


In [11]:
# Create DataFrame with predictions
results_df = pd.DataFrame({
    'Text': X_test,
    'Actual': y_test,
    'Predicted': y_pred
})

# Filter and display misclassified examples
print("\nMisclassified Examples:")
misclassified = results_df[results_df['Actual'] != results_df['Predicted']]
print(misclassified.head(3))




Misclassified Examples:
                                                   Text Actual Predicted
1731  whenever u . s . government starts meddling in...    neg       pos
584   a silly film that tries to be a black comedy b...    neg       pos
1179  director : penelope spheeris ( decline of west...    pos       neg
